In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
from dataclasses import dataclass

import numpy as np
from IPython.display import display
from maite_datasets.image_classification import MilitaryVehicles

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.data import Indices, View
from dataeval.protocols import DatasetMetadata, DatumMetadata
from dataeval.quality import Duplicates

set_max_processes(4)

In [3]:
train_ds = MilitaryVehicles("./data", image_set="train", download=True, lazy=True)
index2label = train_ds.metadata["index2label"]

# Get the labels
labels = Metadata(train_ds).class_labels

In [4]:
# Initialize the Duplicates class to begin to identify duplicate images. A hash radius of 0 counts two images as
# duplicates only when their hashes are identical.
identifyDuplicates = Duplicates(hash_radius=0)

# Evaluate the data
results = identifyDuplicates.evaluate(train_ds)

In [5]:
display(results)

shape: (105, 7)
┌──────────┬───────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods    ┆ differs_on ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---        ┆ ---        ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]  ┆ list[str]  ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪════════════╪════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [478, 6118]  ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 1        ┆ item  ┆ exact    ┆ [487, 7811]  ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 2        ┆ item  ┆ exact    ┆ [508, 3725]  ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 3        ┆ item  ┆ exact    ┆ [520, 7613]  ┆ ["xxhash"] ┆ null       ┆ null                  │
│ 4        ┆ item  ┆ exact    ┆ [544, 7615]  ┆ ["xxhash"] ┆ null       ┆ null                  │
│ …        ┆ …

In [6]:
different = sum(len({labels[i] for i in group}) > 1 for group in results.exact)
first_group = results.exact[0]
print(f"{len(results.exact)} exact groups and {len(results.near)} near groups in {len(train_ds):,} images")
print(f"Exact groups whose images carry different labels: {different} of {len(results.exact)}")
print(f"The first exact group, images {first_group}, is filed under {[index2label[labels[i]] for i in first_group]}")

67 exact groups and 38 near groups in 7,823 images
Exact groups whose images carry different labels: 67 of 67
The first exact group, images [478, 6118], is filed under ['BM-30', 'T-64']


In [7]:
# TEST ASSERTION CELL ###
assert len(train_ds) == 7823
assert (len(results.exact), len(results.near)) == (67, 38)
assert different == len(results.exact)  # the text says every exact group disagrees on the label
assert first_group == [478, 6118]
assert [index2label[labels[i]] for i in first_group] == ["BM-30", "T-64"]
assert all(
    np.array_equal(np.asarray(train_ds[group[0]][0]), np.asarray(train_ds[i][0]))
    for group in results.exact
    for i in group
)  # identical pixels

In [8]:
plan = results.deduplicate()
deduplicated_ds = View(train_ds, Indices(plan, exclude=True))
print(plan)
print(f"{len(train_ds)} items before, {len(deduplicated_ds)} after")

RemovalPlan(items=72)
7823 items before, 7751 after


In [9]:
kept = {
    keep: set(View(train_ds, Indices(results.deduplicate(keep=keep), exclude=True)).resolve_indices())
    for keep in ("first", "last")
}
for keep, survivors in kept.items():
    print(f'keep="{keep}" keeps', [index2label[labels[i]] for i in first_group if i in survivors])

keep="first" keeps ['BM-30']
keep="last" keeps ['T-64']


In [10]:
# TEST ASSERTION CELL ###
assert sum(len(group) for group in results.exact) - len(results.exact) == len(plan) == 72
assert len(deduplicated_ds) == len(train_ds) - len(plan) == 7751
assert [index2label[labels[i]] for i in first_group if i in kept["first"]] == ["BM-30"]
assert [index2label[labels[i]] for i in first_group if i in kept["last"]] == ["T-64"]

In [11]:
@dataclass
class BoxTarget:
    """A minimal object detection target: boxes, labels, and scores."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray


class TinyObjectDetectionDataset:
    """Four items as two duplicate pairs: one relabeled, one augmented."""

    def __init__(self, images: list[np.ndarray], boxes: list[list[list[float]]], labels: list[list[int]]) -> None:
        self._images = images
        self._boxes = boxes
        self._labels = labels
        self.metadata: DatasetMetadata = DatasetMetadata(id="tiny-od", index2label={0: "object"})

    def __len__(self) -> int:
        return len(self._images)

    def __getitem__(self, index: int) -> tuple[np.ndarray, BoxTarget, DatumMetadata]:
        item_labels = np.asarray(self._labels[index], dtype=np.intp)
        target = BoxTarget(
            boxes=np.asarray(self._boxes[index], dtype=np.float32),
            labels=item_labels,
            scores=np.ones(len(item_labels), dtype=np.float32),
        )
        return self._images[index], target, DatumMetadata(id=index)


rng = np.random.default_rng(0)
same_image = rng.random((3, 16, 16))

annotation_images = [same_image, same_image.copy(), rng.random((3, 16, 16)), rng.random((3, 16, 16))]
annotation_boxes = [[[1.0, 1.0, 6.0, 6.0]], [[9.0, 9.0, 14.0, 14.0]], [[0.0, 0.0, 5.0, 5.0]], [[0.0, 0.0, 5.0, 5.0]]]
annotation_labels = [[0], [0], [0], [0]]

annotation_ds = TinyObjectDetectionDataset(annotation_images, annotation_boxes, annotation_labels)

In [12]:
annotation_results = Duplicates(hash_radius=0).evaluate(annotation_ds)

In [13]:
display(annotation_results.divergent())

shape: (1, 7)
┌──────────┬───────┬──────────┬──────────────┬────────────┬────────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods    ┆ differs_on     ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---        ┆ ---            ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]  ┆ list[str]      ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪════════════╪════════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [0, 1]       ┆ ["xxhash"] ┆ ["annotation"] ┆ 1.0                   │
└──────────┴───────┴──────────┴──────────────┴────────────┴────────────────┴───────────────────────┘

In [14]:
display(annotation_results.augmented())

shape: (1, 7)
┌──────────┬───────┬────────────┬──────────────┬────────────────┬─────────────┬────────────────────┐
│ group_id ┆ level ┆ dup_type   ┆ item_indices ┆ methods        ┆ differs_on  ┆ annotation_diverge │
│ ---      ┆ ---   ┆ ---        ┆ ---          ┆ ---            ┆ ---         ┆ nce                │
│ i64      ┆ str   ┆ str        ┆ list[i64]    ┆ list[str]      ┆ list[str]   ┆ ---                │
│          ┆       ┆            ┆              ┆                ┆             ┆ f64                │
╞══════════╪═══════╪════════════╪══════════════╪════════════════╪═════════════╪════════════════════╡
│ 1        ┆ item  ┆ annotation ┆ [2, 3]       ┆ ["annotation"] ┆ ["content"] ┆ null               │
└──────────┴───────┴────────────┴──────────────┴────────────────┴─────────────┴────────────────────┘

In [15]:
# TEST ASSERTION CELL ###
assert len(annotation_results.divergent()) == 1
assert annotation_results.divergent().data()["item_indices"].item().to_list() == [0, 1]
assert len(annotation_results.augmented()) == 1
assert annotation_results.augmented().data()["item_indices"].item().to_list() == [2, 3]